In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding="ISO-8859-1")

In [7]:
df.shape

(1067371, 8)

In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [9]:
df = df.rename(columns={
    "Customer ID" : "CustomerID"
})

In [10]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country'],
      dtype='str')

In [11]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
CustomerID     float64
Country            str
dtype: object

In [12]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [13]:
df["CustomerID"] = df["CustomerID"].astype("Int64")

In [14]:
df.duplicated().sum()

np.int64(34335)

In [15]:
df = df.drop_duplicates()

In [16]:
df["Description"].isnull().sum()

np.int64(4275)

In [17]:
df[df["Description"].isnull()][
    ["Invoice", "StockCode", "Quantity", "Price", "CustomerID"]].head(20)


,Invoice,StockCode,Quantity,Price,CustomerID
470,489521,21646,-50,0.0,<NA>
3114,489655,20683,-44,0.0,<NA>
3161,489659,21350,230,0.0,<NA>
3731,489781,84292,17,0.0,<NA>
4296,489806,18010,-770,0.0,<NA>
4566,489821,85049G,-240,0.0,<NA>
6378,489882,35751C,12,0.0,<NA>
6555,489898,79323G,954,0.0,<NA>
6576,489901,21098,-200,0.0,<NA>
6581,489903,21166,48,0.0,<NA>


In [18]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country'],
      dtype='str')

In [19]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [20]:
df["InvoiceType"] = "Sale"

In [21]:
df.loc[
    df["Invoice"].str.startswith("C"),
    "InvoiceType"
] = "Return"

In [22]:
df.loc[
    df["Invoice"].str.startswith("A"),
    "InvoiceType"
    ] = "Adjustment"

In [23]:
df["InvoiceType"].value_counts()

InvoiceType
Sale          1013926
Return          19104
Adjustment          6
Name: count, dtype: int64

In [24]:
df[df["InvoiceType"] == "Adjustment"][
    ["Invoice", "Description"]
]

,Invoice,Description
179403,A506401,Adjust bad debt
276274,A516228,Adjust bad debt
403472,A528059,Adjust bad debt
825443,A563185,Adjust bad debt
825444,A563186,Adjust bad debt
825445,A563187,Adjust bad debt


In [25]:
df = df[df["InvoiceType"] != "Adjustment"].copy()

In [26]:
df["InvoiceType"].value_counts()

InvoiceType
Sale      1013926
Return      19104
Name: count, dtype: int64

In [27]:
df[df["Quantity"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,InvoiceType
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,Return
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,Return
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,Return
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,Return
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,Return
...,...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397,United Kingdom,Return
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498,United Kingdom,Return
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311,United Kingdom,Return
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315,United Kingdom,Return


In [28]:
pd.crosstab(
    df["InvoiceType"],
    df["Quantity"] < 0
)

Quantity,False,True
InvoiceType,,
Return,1,19103
Sale,1010533,3393


In [29]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [30]:
negative_price = df[df["Price"] < 0]

negative_price[
    ["Invoice", "Description", "Quantity", "Price", "CustomerID"]
].head(20)

,Invoice,Description,Quantity,Price,CustomerID


In [31]:
negative_price["InvoiceType"].value_counts()

Series([], Name: count, dtype: int64)

In [32]:
(df["Price"] < 0).sum()

np.int64(0)

In [33]:
print("Negative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())

Negative prices: 0
Zero prices: 6014
Negative quantities: 22496
Zero quantities: 0


In [34]:
df.loc[df["Price"] == 0, "Description"].value_counts().head(20)

Description
check                            160
?                                 90
damages                           83
damaged                           81
found                             28
missing                           27
sold as set on dotcom             20
Damaged                           17
adjustment                        16
OWL DOORSTOP                      14
POLYESTER FILLER PAD 45x45cm      12
dotcom                            12
POLYESTER FILLER PAD 40x40cm      10
smashed                            9
Found                              9
FRENCH BLUE METAL DOOR SIGN 1      9
thrown away                        9
Unsaleable, destroyed.             9
PICNIC BASKET WICKER LARGE         8
checked                            8
Name: count, dtype: int64

In [35]:
df.loc[
    df["Price"] == 0,
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "InvoiceType"]
].head(20)

,Invoice,StockCode,Description,Quantity,Price,InvoiceType
263,489464,21733,85123a mixed,-96,0.0,Sale
283,489463,71477,short,-240,0.0,Sale
284,489467,85123A,21733 mixed,-192,0.0,Sale
470,489521,21646,NaN,-50,0.0,Sale
3114,489655,20683,NaN,-44,0.0,Sale
3161,489659,21350,NaN,230,0.0,Sale
3162,489660,35956,lost,-1043,0.0,Sale
3168,489663,35605A,damages,-117,0.0,Sale
3731,489781,84292,NaN,17,0.0,Sale
4296,489806,18010,NaN,-770,0.0,Sale


In [36]:
df.loc[
    df["Price"] == 0,
    "InvoiceType"
].value_counts()

InvoiceType
Sale    6014
Name: count, dtype: int64

In [37]:
zero_price  = df[df["Price"] == 0]

print("Zero-price rows", len(zero_price))
print("Zero-price quantity:", zero_price["Quantity"].sum())

Zero-price rows 6014
Zero-price quantity: -318555


In [38]:
zero_price["Quantity"].describe()

count     6014.000000
mean       -52.968906
std        642.016002
min      -9600.000000
25%        -33.000000
50%         -3.000000
75%          3.000000
max      12540.000000
Name: Quantity, dtype: float64

In [39]:
clean_df = df[df["Price"] > 0].copy()

In [40]:
print("Original rows:", len(df))
print("Clean rows:", len(clean_df))
print("Removed rows:", len(df) - len(clean_df))

Original rows: 1033030
Clean rows: 1027016
Removed rows: 6014


In [41]:
print("Negative_prices:", (clean_df["Price"] < 0).sum())
print("Zero_prices:", (clean_df["Price"] == 0).sum())

Negative_prices: 0
Zero_prices: 0


In [42]:
print("Negative_quantity", (clean_df["Quantity"] < 0).sum())
print("Positive_quantity", (clean_df["Quantity"] > 0).sum())

Negative_quantity 19103
Positive_quantity 1007913


In [43]:
pd.crosstab(
    clean_df["InvoiceType"],
    clean_df["Quantity"] < 0
)

Quantity,False,True
InvoiceType,,
Return,1,19103
Sale,1007912,0


In [44]:
clean_df["Revenue"] = (
    clean_df["Quantity"] * clean_df["Price"]
)

In [45]:
clean_df[["Quantity", "Price", "Revenue"]].head(10)

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0
5,24,1.65,39.6
6,24,1.25,30.0
7,10,5.95,59.5
8,12,2.55,30.6
9,12,3.75,45.0


In [46]:
clean_df.loc[
    clean_df["Quantity"] <0,
    ["Quantity", "Price", "Revenue"]
].head(10)

,Quantity,Price,Revenue
178,-12,2.95,-35.40
179,-6,1.65,-9.90
180,-4,4.25,-17.00
181,-6,2.10,-12.60
182,-12,2.95,-35.40
183,-12,1.25,-15.00
184,-12,1.25,-15.00
185,-24,0.85,-20.40
186,-12,2.95,-35.40
196,-3,4.25,-12.75


In [47]:
clean_df["Revenue"].sum()

np.float64(19003147.778000005)

In [48]:
sales_rev = clean_df.loc[
    clean_df["InvoiceType"] == "Sale", "Revenue"
].sum()

return_rev = clean_df.loc[
    clean_df["InvoiceType"] == "Return", "Revenue"
].sum()

print(sales_rev)
print(return_rev)

print(sales_rev  + return_rev)  

20465198.387999997
-1462050.61
19003147.777999997


In [49]:
clean_df.isnull().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
CustomerID     229201
Country             0
InvoiceType         0
Revenue             0
dtype: int64

In [50]:
clean_df["CustomerID"].isnull().mean()*100

np.float64(22.317179089712333)

In [51]:
customer_df = clean_df[
    clean_df["CustomerID"].notnull()
].copy()

print("Original rows:", len(clean_df))
print("Rows with valid CustomerID:", len(customer_df))

Original rows: 1027016
Rows with valid CustomerID: 797815


In [52]:
print("Missing CustomerID:",
      clean_df["CustomerID"].isna().sum())

print("Known CustomerID:",
      clean_df["CustomerID"].notna().sum())

Missing CustomerID: 229201
Known CustomerID: 797815


In [53]:
clean_df.duplicated().sum()

np.int64(0)

In [54]:
description_counts = (
    clean_df.groupby("StockCode")["Description"]
    .nunique()
    .sort_values(ascending=False)
)

print(description_counts.head(10))

StockCode
22344    4
22345    4
23196    4
21955    4
22346    4
20685    4
23236    4
22384    4
22139    3
22952    3
Name: Description, dtype: int64


In [55]:
clean_df["InvoiceDate"].dtype

dtype('<M8[us]')

In [56]:
min_date = clean_df["InvoiceDate"].min()
max_date = clean_df["InvoiceDate"].max()

print(min_date)
print(max_date)

2009-12-01 07:45:00
2011-12-09 12:50:00


In [57]:
clean_df["CustomerID"].nunique()

5939

In [58]:
clean_df["CustomerID"].value_counts().head(10)

CustomerID
17841    12638
14911    11442
12748     6660
14606     6500
14096     5128
15311     4579
14156     4118
14646     3885
13089     3390
16549     3098
Name: count, dtype: Int64

In [59]:
clean_df["Country"].nunique()

43

In [60]:
clean_df["Country"].value_counts().head(15)

Country
United Kingdom     942328
EIRE                17662
Germany             17331
France              14024
Netherlands          5132
Spain                3753
Switzerland          3174
Belgium              3109
Portugal             2528
Australia            1887
Channel Islands      1646
Italy                1507
Sweden               1362
Norway               1307
Cyprus               1157
Name: count, dtype: int64

In [61]:
clean_df.groupby("Country")["Revenue"].sum()

Country
Australia               1.664444e+05
Austria                 2.317760e+04
Bahrain                 2.861550e+03
Belgium                 6.320889e+04
Bermuda                 1.253140e+03
Brazil                  1.411870e+03
Canada                  4.883040e+03
Channel Islands         4.108018e+04
Cyprus                  2.403256e+04
Czech Republic          7.077200e+02
Denmark                 6.445959e+04
EIRE                    6.099538e+05
European Community      1.291750e+03
Finland                 2.951445e+04
France                  3.217334e+05
Germany                 4.119592e+05
Greece                  1.899549e+04
Hong Kong               1.383050e+04
Iceland                 4.921530e+03
Israel                  1.110137e+04
Italy                   3.025410e+04
Japan                   3.966210e+04
Korea                   9.498200e+02
Lebanon                 1.865910e+03
Lithuania               4.892680e+03
Malta                   5.192220e+03
Netherlands             5.4833

In [62]:
clean_df["Country"].str.strip().nunique()

43

In [63]:
clean_df.duplicated().sum()

np.int64(0)

In [64]:
invoice_size = clean_df.groupby("Invoice").size()
invoice_size.describe()

count    48368.000000
mean        21.233377
std         39.681446
min          1.000000
25%          3.000000
50%         11.000000
75%         25.000000
max       1114.000000
dtype: float64

In [65]:
clean_df["Invoice"].nunique()

48368

In [66]:
clean_df.groupby("InvoiceType")["Invoice"].nunique()

InvoiceType
Return     8292
Sale      40076
Name: Invoice, dtype: int64

In [67]:
clean_df.groupby("InvoiceType")["Quantity"].agg(["count", "sum"])

,count,sum
InvoiceType,,
Return,19104,-476819
Sale,1007912,11205147


In [68]:
net_quantity = clean_df["Quantity"].sum()

print(net_quantity)

10728328


In [69]:
clean_df["Revenue"].describe()


count    1.027016e+06
mean     1.850326e+01
std      2.853397e+02
min     -1.684696e+05
25%      3.750000e+00
50%      9.950000e+00
75%      1.770000e+01
max      1.684696e+05
Name: Revenue, dtype: float64

In [70]:
clean_df.nlargest(10, "Revenue")[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Revenue", "InvoiceType"]
]

,Invoice,StockCode,Description,Quantity,Price,Revenue,InvoiceType
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,Sale
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,Sale
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00,Sale
241827,512771,M,Manual,1,25111.09,25111.09,Sale
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.40,Sale
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33,Sale
135013,502263,M,Manual,1,10953.50,10953.50,Sale
135015,502265,M,Manual,1,10953.50,10953.50,Sale
342147,522796,M,Manual,1,10468.80,10468.80,Sale
358639,524159,M,Manual,1,10468.80,10468.80,Sale


In [71]:
clean_df.nsmallest(10, "Revenue")[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Revenue", "InvoiceType"]
]

,Invoice,StockCode,Description,Quantity,Price,Revenue,InvoiceType
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,-168469.60,Return
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,-77183.60,Return
748142,C556445,M,Manual,-1,38970.00,-38970.00,Return
241824,C512770,M,Manual,-1,25111.09,-25111.09,Return
320581,C520667,BANK CHARGES,Bank Charges,-1,18910.69,-18910.69,Return
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,17836.46,-17836.46,Return
569163,C540117,AMAZONFEE,AMAZON FEE,-1,16888.02,-16888.02,Return
569164,C540118,AMAZONFEE,AMAZON FEE,-1,16453.71,-16453.71,Return
517953,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33,-13541.33,Return
519294,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33,-13541.33,Return


In [72]:
clean_df.nlargest(
    20,
    "Quantity"
)[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Revenue", "InvoiceType"]
]

,Invoice,StockCode,Description,Quantity,Price,Revenue,InvoiceType
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,Sale
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,Sale
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,Sale
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,Sale
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,Sale
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,Sale
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,Sale
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,Sale
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,Sale
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,Sale


In [73]:
clean_df.nsmallest(
    20,
    "Quantity"
)[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Revenue", "InvoiceType"]
]

,Invoice,StockCode,Description,Quantity,Price,Revenue,InvoiceType
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,-168469.60,Return
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,-77183.60,Return
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,-280.80,Return
359669,C524235,21088,SET/6 FRUIT SALAD PAPER CUPS,-7128,0.08,-570.24,Return
359670,C524235,21096,SET/6 FRUIT SALAD PAPER PLATES,-7008,0.13,-911.04,Return
359630,C524235,16047,POP ART PEN CASE & PENS,-5184,0.08,-414.72,Return
359636,C524235,37340,MULTICOLOUR SPRING FLOWER MUG,-4992,0.10,-499.20,Return
359653,C524235,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,-4752,0.07,-332.64,Return
359658,C524235,16046,TEATIME PEN CASE & PENS,-4608,0.08,-368.64,Return
359654,C524235,85160A,WHITE BIRD GARDEN DESIGN MUG,-4320,0.13,-561.60,Return


In [74]:
clean_df[
    ~clean_df["StockCode"].astype(str).str.match(r"^\d")
]["StockCode"].value_counts()

StockCode
POST            2079
DOT             1418
M               1380
C2               274
D                173
S                101
BANK CHARGES     100
ADJUST            67
AMAZONFEE         36
DCGS0058          30
gift_0001_20      26
gift_0001_30      24
DCGSSGIRL         23
DCGSSBOY          21
PADS              18
CRUK              16
DCGS0076          14
gift_0001_10      14
DCGS0003          13
TEST001           13
gift_0001_50       6
m                  5
DCGS0069           5
gift_0001_40       5
DCGS0004           4
DCGS0072           3
ADJUST2            3
DCGS0068           2
gift_0001_80       2
DCGS0066N          2
DCGS0070           2
SP1002             2
DCGS0044           1
TEST002            1
DCGS0075           1
DCGS0041           1
gift_0001_70       1
DCGS0037           1
DCGS0062           1
Name: count, dtype: int64

In [75]:
analysis_df = clean_df[
    clean_df["StockCode"].astype(str).str.match(r"^\d")
].copy()

In [76]:
analysis_df = clean_df[
    clean_df["StockCode"].astype(str).str.match(r"^\d")
].copy()

print("Original rows:", len(clean_df))
print("Analysis rows:", len(analysis_df))
print("Rows excluded:", len(clean_df) - len(analysis_df))

Original rows: 1027016
Analysis rows: 1021128
Rows excluded: 5888


In [77]:
print("Rows:", len(analysis_df))
print("Columns:", analysis_df.shape[1])

print("\nMissing values:")
print(analysis_df.isna().sum())

print("\nInvoice types:")
print(analysis_df["InvoiceType"].value_counts())

print("\nNegative quantities:")
print((analysis_df["Quantity"] < 0).sum())

print("\nNegative prices:")
print((analysis_df["Price"] < 0).sum())

Rows: 1021128
Columns: 10

Missing values:
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
CustomerID     226965
Country             0
InvoiceType         0
Revenue             0
dtype: int64

Invoice types:
InvoiceType
Sale      1003214
Return      17914
Name: count, dtype: int64

Negative quantities:
17914

Negative prices:
0


In [78]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [79]:
clean_df.to_csv(
    "../data/processed/clean_retail.csv",
    index = False
)
analysis_df.to_csv(
    "../data/processed/analysis_retail.csv",
    index=False
)

In [80]:
analysis_df["StockCode"].nunique()

4892

In [81]:
dim_product = (
    analysis_df[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .reset_index(drop=True)
)

In [82]:
dim_product["ProductKey"] = range(1,len(dim_product) + 1)

In [83]:
dim_product = dim_product[
    ["ProductKey", "StockCode", "Description"]
]

In [84]:
dim_product.head()

,ProductKey,StockCode,Description
0,1,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,2,79323P,PINK CHERRY LIGHTS
2,3,79323W,WHITE CHERRY LIGHTS
3,4,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,5,21232,STRAWBERRY CERAMIC TRINKET BOX


In [85]:
print("Products:", len(dim_product))
print("Unique StockCodes:", dim_product["StockCode"].nunique())
print("Duplicate StockCodes:", dim_product["StockCode"].duplicated().sum())

Products: 4892
Unique StockCodes: 4892
Duplicate StockCodes: 0


In [86]:
analysis_df["CustomerID"].nunique()

5875

In [87]:
analysis_df["CustomerID"].isnull().sum()

np.int64(226965)

In [88]:
dim_customer = (
    analysis_df[
        ["CustomerID", "Country"]
    ].dropna(subset=["CustomerID"]).drop_duplicates(subset=["CustomerID"])
    .reset_index(drop=True)
)

In [89]:
dim_customer["CustomerKey"] = range(
    1,len(dim_customer) +1
)

In [90]:
dim_customer = dim_customer[
    ["CustomerKey", "CustomerID", "Country"]
]

In [91]:
dim_customer.head()

,CustomerKey,CustomerID,Country
0,1,13085,United Kingdom
1,2,13078,United Kingdom
2,3,15362,United Kingdom
3,4,18102,United Kingdom
4,5,12682,France


In [92]:
print("Customers:", len(dim_customer))
print("Unique CustomerIDs:", dim_customer["CustomerID"].nunique())
print("Duplicate CustomerIDs:", dim_customer["CustomerID"].duplicated().sum())

Customers: 5875
Unique CustomerIDs: 5875
Duplicate CustomerIDs: 0


In [93]:
analysis_df["InvoiceDate"].head()

0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
Name: InvoiceDate, dtype: datetime64[us]

In [94]:
analysis_df["InvoiceDate"].dtype

dtype('<M8[us]')

In [95]:
date_series = (
    analysis_df["InvoiceDate"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
)

In [96]:
dim_date = pd.DataFrame({
    "Date": date_series
})

In [97]:
dim_date.head()

,Date
0,2009-12-01
3223,2009-12-02
6500,2009-12-03
9502,2009-12-04
12061,2009-12-05


In [98]:
dim_date["DateKey"] = (
    dim_date["Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [99]:
dim_date["Year"] = dim_date["Date"].dt.year

In [100]:
dim_date["Quarter"] = (
    "Q" + dim_date["Date"].dt.quarter.astype(str)
)

In [101]:
dim_date["Month"] = dim_date["Date"].dt.month

In [102]:
dim_date["MonthName"] = dim_date["Date"].dt.month_name()

In [103]:
dim_date["Day"] = dim_date["Date"].dt.day

In [104]:
dim_date["Week"] = dim_date["Date"].dt.isocalendar().week.astype(int)


In [105]:
dim_date = dim_date[
    [
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "Month",
        "MonthName",
        "Day",
        "Week"
    ]
]

In [106]:
dim_date.head(10)

,DateKey,Date,Year,Quarter,Month,MonthName,Day,Week
0,20091201,2009-12-01,2009,Q4,12,December,1,49
3223,20091202,2009-12-02,2009,Q4,12,December,2,49
6500,20091203,2009-12-03,2009,Q4,12,December,3,49
9502,20091204,2009-12-04,2009,Q4,12,December,4,49
12061,20091205,2009-12-05,2009,Q4,12,December,5,49
12463,20091206,2009-12-06,2009,Q4,12,December,6,49
14405,20091207,2009-12-07,2009,Q4,12,December,7,50
17274,20091208,2009-12-08,2009,Q4,12,December,8,50
19714,20091209,2009-12-09,2009,Q4,12,December,9,50
22184,20091210,2009-12-10,2009,Q4,12,December,10,50


In [107]:
analysis_df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country', 'InvoiceType', 'Revenue'],
      dtype='str')

In [108]:
analysis_df["DateKey"] = (
    analysis_df["InvoiceDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [109]:
product_map = dim_product.set_index("StockCode")["ProductKey"]

In [110]:
analysis_df["ProductKey"] = analysis_df["StockCode"].map(product_map)

In [111]:
customer_map = dim_customer.set_index("CustomerID")["CustomerKey"]

In [112]:
analysis_df["CustomerKey"] = analysis_df["CustomerID"].map(customer_map)

In [113]:
analysis_df["CustomerKey"] = (
    analysis_df["CustomerKey"]
    .fillna(0)
    .astype(int)
)

In [114]:
analysis_df["CustomerKey"].isna().sum()

np.int64(0)

In [115]:
analysis_df["ProductKey"].isna().sum()

np.int64(0)

In [116]:
unknown_customer = pd.DataFrame({
    "CustomerKey": [0],
    "CustomerID": [pd.NA],
    "Country": ["Unknown"]
})

dim_customer = pd.concat(
    [unknown_customer, dim_customer],
    ignore_index=True
)

In [117]:
fact_sales = analysis_df[
    [
        "Invoice",
        "ProductKey",
        "CustomerKey",
        "DateKey",
        "Quantity",
        "Price",
        "Revenue",
        "InvoiceType"
    ]
].copy()

In [118]:
fact_sales["SaleKey"] = range(1,len(fact_sales)+1)

In [119]:
print(len(fact_sales))
print(len(analysis_df))

1021128
1021128


In [120]:
fact_sales["SaleKey"].duplicated().sum()

np.int64(0)

In [121]:
~fact_sales["ProductKey"].isin(dim_product["ProductKey"]).sum()

np.int64(-1021129)

In [122]:
print(
    "Invalid ProductKeys:",
    (~fact_sales["ProductKey"].isin(dim_product["ProductKey"])).sum()
)

print(
    "Invalid CustomerKeys:",
    (~fact_sales["CustomerKey"].isin(dim_customer["CustomerKey"])).sum()
)

print(
    "Invalid DateKeys:",
    (~fact_sales["DateKey"].isin(dim_date["DateKey"])).sum()
)

Invalid ProductKeys: 0
Invalid CustomerKeys: 0
Invalid DateKeys: 0


In [123]:
print("dim_product:", dim_product.shape)
print("dim_customer:", dim_customer.shape)
print("dim_date:", dim_date.shape)
print("fact_sales:", fact_sales.shape)

dim_product: (4892, 3)
dim_customer: (5876, 3)
dim_date: (604, 8)
fact_sales: (1021128, 9)


In [124]:
print("\nProduct columns:")
print(dim_product.columns.tolist())

print("\nCustomer columns:")
print(dim_customer.columns.tolist())

print("\nDate columns:")
print(dim_date.columns.tolist())

print("\nFact columns:")
print(fact_sales.columns.tolist())


Product columns:
['ProductKey', 'StockCode', 'Description']

Customer columns:
['CustomerKey', 'CustomerID', 'Country']

Date columns:
['DateKey', 'Date', 'Year', 'Quarter', 'Month', 'MonthName', 'Day', 'Week']

Fact columns:
['Invoice', 'ProductKey', 'CustomerKey', 'DateKey', 'Quantity', 'Price', 'Revenue', 'InvoiceType', 'SaleKey']


In [125]:
fact_sales.loc[fact_sales["InvoiceType"] == "Sale",
"Revenue"].sum()

np.float64(19642692.150000002)

In [126]:
print("Sale Revenue:",
      fact_sales.loc[
          fact_sales["InvoiceType"] == "Sale",
          "Revenue"
      ].sum())

print("Return Revenue:",
      fact_sales.loc[
          fact_sales["InvoiceType"] == "Return",
          "Revenue"
      ].sum())

print("Net Revenue:",
      fact_sales["Revenue"].sum())

Sale Revenue: 19642692.150000002
Return Revenue: -716425.97
Net Revenue: 18926266.18


In [127]:
print("Analysis total:", analysis_df["Revenue"].sum())
print("Fact total:", fact_sales["Revenue"].sum())

Analysis total: 18926266.18
Fact total: 18926266.18


In [128]:
print("Fact rows:", len(fact_sales))

print(
    fact_sales.groupby("InvoiceType")["Revenue"].agg(["count", "sum"])
)

print(
    analysis_df.groupby("InvoiceType")["Revenue"].agg(["count", "sum"])
)

print("Analysis total:", analysis_df["Revenue"].sum())
print("Fact total:", fact_sales["Revenue"].sum())

Fact rows: 1021128


               count          sum
InvoiceType                      
Return         17914   -716425.97
Sale         1003214  19642692.15
               count          sum
InvoiceType                      
Return         17914   -716425.97
Sale         1003214  19642692.15
Analysis total: 18926266.18
Fact total: 18926266.18


# importing csv to neon

In [129]:
import os
os.makedirs("../data/processed",exist_ok=True)

In [130]:
dim_product.to_csv(
    "../data/processed/dim_product.csv",
    index = False
)

dim_customer.to_csv(
    "../data/processed/dim_customer.csv",
    index = False
)

dim_date.to_csv(
    "../data/processed/dim_date.csv",
    index = False
)

fact_sales.to_csv(
    "../data/processed/fact_sales.csv",
    index =False
)

In [131]:
import os
processed_path = "../data/processed"

print(os.listdir(processed_path))

['analysis_retail.csv', 'clean_retail.csv', 'dim_customer.csv', 'dim_date.csv', 'dim_product.csv', 'fact_sales.csv']


In [132]:
for file in os.listdir(processed_path):
    path = os.path.join(processed_path, file)
    print(file,os.path.getsize(path) / (1024*1024), "MB" )

analysis_retail.csv 98.23663806915283 MB
clean_retail.csv 98.6787281036377 MB
dim_customer.csv 0.14689922332763672 MB
dim_date.csv 0.025358200073242188 MB
dim_product.csv 0.18903064727783203 MB
fact_sales.csv 50.65033435821533 MB


In [133]:
pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [134]:
pip install python dotenv


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement python (from versions: none)
ERROR: No matching distribution found for python


In [135]:
pip install sqlalchemy psycopg2-binary python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [136]:
import sqlalchemy
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")


In [137]:
from sqlalchemy import create_engine, text

engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.fetchone()) 

(1,)


In [138]:
from sqlalchemy import text

with engine.begin() as connection:

    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_product (
            ProductKey INTEGER PRIMARY KEY,
            StockCode TEXT,
            Description TEXT
        );
    """))

print("dim_product created")

dim_product created


In [139]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_customer (
            CustomerKey INTEGER PRIMARY KEY,
            CustomerID FLOAT,
            Country TEXT
        );
    """))

print("dim_customer created")

dim_customer created


In [140]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS dim_date (
            DateKey INTEGER PRIMARY KEY,
            Date DATE,
            Year INTEGER,
            Quarter INTEGER,
            Month INTEGER,
            MonthName TEXT,
            Day INTEGER,
            Week INTEGER
        );
    """))

print("dim_date created")

dim_date created


In [141]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS fact_sales (
            SaleKey INTEGER PRIMARY KEY,
            Invoice TEXT,
            ProductKey INTEGER,
            CustomerKey INTEGER,
            DateKey INTEGER,
            Quantity INTEGER,
            Price NUMERIC,
            Revenue NUMERIC,
            InvoiceType TEXT
        );
    """))

print("fact_sales created")

fact_sales created


In [142]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))

    for row in result:
        print(row[0])

dim_customer
dim_date
dim_product
fact_sales


In [143]:
import pandas as pd

dim_product = pd.read_csv("../data/processed/dim_product.csv")

print(dim_product.shape)
print(dim_product.head())

(4892, 3)
   ProductKey StockCode                          Description
0           1     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS
1           2    79323P                   PINK CHERRY LIGHTS
2           3    79323W                  WHITE CHERRY LIGHTS
3           4     22041         RECORD FRAME 7" SINGLE SIZE 
4           5     21232       STRAWBERRY CERAMIC TRINKET BOX


In [145]:
dim_product.to_sql(
    "dim_product",
    engine,
    if_exists="append",
    index = False
)

DatabaseError: Execution failed on sql 'INSERT INTO dim_product ("ProductKey", "StockCode", "Description") VALUES (:ProductKey, :StockCode, :Description)': (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "dim_product_pkey"
DETAIL:  Key ("ProductKey")=(1) already exists.

[SQL: INSERT INTO dim_product ("ProductKey", "StockCode", "Description") VALUES (%(ProductKey__0)s, %(StockCode__0)s, %(Description__0)s), (%(ProductKey__1)s, %(StockCode__1)s, %(Description__1)s), (%(ProductKey__2)s, %(StockCode__2)s, %(Description__2)s), ... 64392 characters truncated ... kCode__998)s, %(Description__998)s), (%(ProductKey__999)s, %(StockCode__999)s, %(Description__999)s)]
[parameters: {'StockCode__0': '85048', 'Description__0': '15CM CHRISTMAS GLASS BALL 20 LIGHTS', 'ProductKey__0': 1, 'StockCode__1': '79323P', 'Description__1': 'PINK CHERRY LIGHTS', 'ProductKey__1': 2, 'StockCode__2': '79323W', 'Description__2': ' WHITE CHERRY LIGHTS', 'ProductKey__2': 3, 'StockCode_

In [146]:
from sqlalchemy import text

with engine.begin() as connection:
    connection.execute(text("DROP TABLE IF EXISTS fact_sales"))
    connection.execute(text("DROP TABLE IF EXISTS dim_product"))
    connection.execute(text("DROP TABLE IF EXISTS dim_customer"))
    connection.execute(text("DROP TABLE IF EXISTS dim_date"))

print("Old tables deleted")

Old tables deleted


In [147]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE dim_product (
            "ProductKey" INTEGER PRIMARY KEY,
            "StockCode" TEXT,
            "Description" TEXT
        );
    """))

print("dim_product recreated")


dim_product recreated


In [148]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE dim_customer (
            "CustomerKey" INTEGER PRIMARY KEY,
            "CustomerID" FLOAT,
            "Country" TEXT
        );
    """))

    connection.execute(text("""
        CREATE TABLE dim_date (
            "DateKey" INTEGER PRIMARY KEY,
            "Date" DATE,
            "Year" INTEGER,
            "Quarter" INTEGER,
            "Month" INTEGER,
            "MonthName" TEXT,
            "Day" INTEGER,
            "Week" INTEGER
        );
    """))

    connection.execute(text("""
        CREATE TABLE fact_sales (
            "SaleKey" INTEGER PRIMARY KEY,
            "Invoice" TEXT,
            "ProductKey" INTEGER,
            "CustomerKey" INTEGER,
            "DateKey" INTEGER,
            "Quantity" INTEGER,
            "Price" NUMERIC,
            "Revenue" NUMERIC,
            "InvoiceType" TEXT
        );
    """))

print("All tables recreated")

All tables recreated


In [149]:
dim_product.to_sql(
    "dim_product",
    engine,
    if_exists="append",
    index = False
)

892

In [150]:
dim_customer = pd.read_csv("../data/processed/dim_customer.csv")

In [151]:
dim_customer.to_sql(
    "dim_customer",
    engine,
    if_exists= "append",
    index = False
)

876

In [152]:
dim_date = pd.read_csv("../data/processed/dim_date.csv")

In [153]:
from sqlalchemy import text

with engine.begin() as connection:
    connection.execute(text("DROP TABLE IF EXISTS dim_date"))

print("dim_date deleted")

dim_date deleted


In [154]:
with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE dim_date (
            "DateKey" INTEGER PRIMARY KEY,
            "Date" DATE,
            "Year" INTEGER,
            "Quarter" TEXT,
            "Month" INTEGER,
            "MonthName" TEXT,
            "Day" INTEGER,
            "Week" INTEGER
        );
    """))

print("dim_date recreated")

dim_date recreated


In [155]:
dim_date.to_sql(
    "dim_date",
    engine,
    if_exists = "append",
    index = False
)

604

In [156]:
fact_sales = pd.read_csv("../data/processed/fact_sales.csv")

In [164]:
fact_sales.to_sql(
    "fact_sales",
    engine,
    if_exists = "append",
    index = False
)

128

# SQL Queries

In [160]:
with engine.connect() as connection:
    result = connection.execute(
        text('SELECT COUNT(*) FROM "fact_sales"')
    )
    print("Rows currently in database:", result.fetchone()[0])

Rows currently in database: 0


In [161]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            current_database(),
            current_schema(),
            current_user;
    """))

    print(result.fetchone())

('neondb', 'public', 'neondb_owner')


In [162]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))

    for row in result:
        print(row[0])

dim_customer
dim_date
dim_product
fact_sales


In [163]:
with engine.connect() as connection:
    for table in ["dim_product", "dim_customer", "dim_date", "fact_sales"]:
        result = connection.execute(
            text(f'SELECT COUNT(*) FROM "{table}"')
        )
        print(table, ":", result.fetchone()[0])

dim_product : 4892
dim_customer : 5876
dim_date : 604
fact_sales : 0


In [159]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS rows,
            SUM("Revenue") AS total_revenue
        FROM fact_sales;
    """))

    print(result.fetchone())

(0, None)


In [165]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM fact_sales f
        LEFT JOIN dim_product p
            ON f."ProductKey" = p."ProductKey"
        WHERE p."ProductKey" IS NULL;
    """))

    print("Invalid ProductKeys:", result.fetchone()[0])

Invalid ProductKeys: 0


In [166]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT COUNT(*)
        FROM fact_sales f
        LEFT JOIN dim_customer c
            ON f."CustomerKey" = c."CustomerKey"
        WHERE c."CustomerKey" IS NULL;
    """))

    print("Invalid CustomerKeys:", result.fetchone()[0])

Invalid CustomerKeys: 0


In [167]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT "ProductKey", COUNT(*)
        FROM dim_product
        GROUP BY "ProductKey"
        HAVING COUNT(*) > 1;
    """))

    rows = result.fetchall()
    print("Duplicate ProductKeys:", len(rows))

Duplicate ProductKeys: 0


In [168]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (WHERE "ProductKey" IS NULL) AS null_product,
            COUNT(*) FILTER (WHERE "CustomerKey" IS NULL) AS null_customer,
            COUNT(*) FILTER (WHERE "DateKey" IS NULL) AS null_date,
            COUNT(*) FILTER (WHERE "Revenue" IS NULL) AS null_revenue
        FROM fact_sales;
    """))

    print(result.fetchone())

(1021128, 0, 0, 0, 0)


In [169]:
with engine.begin() as connection:
    connection.execute(text("""
        ALTER TABLE fact_sales
        DROP CONSTRAINT IF EXISTS fk_product;
    """))

    connection.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_product
        FOREIGN KEY ("ProductKey")
        REFERENCES dim_product ("ProductKey");
    """))

    connection.execute(text("""
        ALTER TABLE fact_sales
        DROP CONSTRAINT IF EXISTS fk_customer;
    """))

    connection.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_customer
        FOREIGN KEY ("CustomerKey")
        REFERENCES dim_customer ("CustomerKey");
    """))

    connection.execute(text("""
        ALTER TABLE fact_sales
        DROP CONSTRAINT IF EXISTS fk_date;
    """))

    connection.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_date
        FOREIGN KEY ("DateKey")
        REFERENCES dim_date ("DateKey");
    """))

print("Foreign-key constraints added successfully")

Foreign-key constraints added successfully


In [173]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            constraint_name,
            constraint_type
        FROM information_schema.table_constraints
        WHERE table_schema = 'public'
          AND table_name = 'fact_sales';
    """))

    for row in result:
        print(row)

('fact_sales_SaleKey_not_null', 'CHECK')
('fact_sales_pkey', 'PRIMARY KEY')
('fk_product', 'FOREIGN KEY')
('fk_customer', 'FOREIGN KEY')
('fk_date', 'FOREIGN KEY')
